# Experiment: CADICA Leakage Audit

**Objective**
- Audit CADICA for deterministic leakage signals against the actual local raw dataset, tracked split preset, expanded split manifest, and prepared YOLO export.
- Separate confirmed leakage findings from confirmed no-leak checks and descriptive risk signals.
- Fail loudly if the current patient-level seed-42 split is inconsistent with the prepared dataset on disk.


In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import re
import struct
import warnings
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

assert importlib.util.find_spec("openpyxl") is not None, (
    "openpyxl is required to read metadata.xlsx. Run `uv sync --project output/jupyter-notebook` "
    "before executing this notebook."
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


## Notebook Map

This notebook builds the requested canonical tables:
- `manifest_split_df`
- `raw_selected_frame_df`
- `prepared_frame_df`
- `metadata_df`
- `audit_df`

It also creates helper tables for duplicate analysis and risk interpretation:
- `video_inventory_df`
- `prepared_duplicate_groups_df`
- `raw_duplicate_groups_df`
- `table_summary_df`


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root from {start}")


def strip_lines(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def natural_suffix(value: str) -> int:
    match = re.search(r"(\d+)$", value)
    if match is None:
        raise ValueError(f"Expected numeric suffix in {value}")
    return int(match.group(1))


def read_png_size(path: Path) -> tuple[int, int]:
    with path.open("rb") as handle:
        header = handle.read(24)
    if header[:8] != b"\x89PNG\r\n\x1a\n":
        raise ValueError(f"Unexpected PNG signature for {path}")
    return struct.unpack(">II", header[16:24])


def md5_file(path: Path) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def count_nonempty_lines(path: Path) -> int:
    return sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())


def resolve_frame_image(input_dir: Path, frame_stem: str) -> Path:
    for suffix in (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"):
        candidate = input_dir / f"{frame_stem}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not resolve image for {frame_stem} in {input_dir}")


REPO_ROOT = find_repo_root(Path.cwd())
RAW_CADICA_ROOT = REPO_ROOT / "datasets" / "cadica" / "CADICA"
TRACKED_SPLIT_MANIFEST = REPO_ROOT / "models" / "yolo26m_cadica" / "manifests" / "patient_level_80_10_10_seed42.json"
EXPANDED_SPLIT_MANIFEST = RAW_CADICA_ROOT / "splits" / "patient_level_80_10_10_seed42" / "manifest.json"
PREPARED_DATASET_ROOT = REPO_ROOT / "datasets" / "cadica" / "derived" / "yolo26_selected_seed42"
SELECTED_ROOT = RAW_CADICA_ROOT / "selectedVideos"
NONSELECTED_ROOT = RAW_CADICA_ROOT / "nonselectedVideos"
METADATA_PATH = RAW_CADICA_ROOT / "metadata.xlsx"
PROJECTION_PATH = SELECTED_ROOT / "CADICAprojections.json"
SPLITS = ("train", "val", "test")
PREPARED_STEM_RE = re.compile(r"^cadica_(p\d+)_(v\d+)_(\d+)$")
EXPECTED_PATIENTS = 42

assert RAW_CADICA_ROOT.exists(), f"Missing raw CADICA root: {RAW_CADICA_ROOT}"
assert TRACKED_SPLIT_MANIFEST.exists(), f"Missing tracked split manifest: {TRACKED_SPLIT_MANIFEST}"
assert EXPANDED_SPLIT_MANIFEST.exists(), f"Missing expanded split manifest: {EXPANDED_SPLIT_MANIFEST}"
assert SELECTED_ROOT.exists(), f"Missing selectedVideos directory: {SELECTED_ROOT}"
assert NONSELECTED_ROOT.exists(), f"Missing nonselectedVideos directory: {NONSELECTED_ROOT}"
assert METADATA_PATH.exists(), f"Missing metadata workbook: {METADATA_PATH}"
assert PROJECTION_PATH.exists(), f"Missing projection mapping: {PROJECTION_PATH}"

prepared_dataset_available = PREPARED_DATASET_ROOT.exists()


In [3]:
tracked_manifest = json.loads(TRACKED_SPLIT_MANIFEST.read_text(encoding="utf-8"))
expanded_manifest = json.loads(EXPANDED_SPLIT_MANIFEST.read_text(encoding="utf-8"))
projections = json.loads(PROJECTION_PATH.read_text(encoding="utf-8"))
projection_lookup = {
    video_key: projection_group
    for projection_group, video_keys in projections.items()
    for video_key in video_keys
}


def manifest_maps(payload: dict, include_selected_videos: bool) -> tuple[dict[str, str], dict[str, str]]:
    patient_to_split: dict[str, str] = {}
    video_to_split: dict[str, str] = {}
    for split in SPLITS:
        split_payload = payload["splits"][split]
        for patient in split_payload["patients"]:
            if patient in patient_to_split:
                raise AssertionError(f"Patient {patient} appears in more than one split")
            patient_to_split[patient] = split
        if include_selected_videos:
            for video_key in split_payload["selected_videos"]:
                if video_key in video_to_split:
                    raise AssertionError(f"Selected video {video_key} appears in more than one split")
                video_to_split[video_key] = split
    return patient_to_split, video_to_split


tracked_patient_to_split, _ = manifest_maps(tracked_manifest, include_selected_videos=False)
expanded_patient_to_split, expanded_video_to_split = manifest_maps(expanded_manifest, include_selected_videos=True)

manifest_rows: list[dict] = []
for split in SPLITS:
    for video_key in sorted(expanded_manifest["splits"][split]["selected_videos"]):
        patient, video = video_key.split("_", 1)
        manifest_rows.append(
            {
                "split": split,
                "patient": patient,
                "video": video,
                "selected_video": video_key,
            }
        )
manifest_split_df = pd.DataFrame(manifest_rows).sort_values(["split", "patient", "video"]).reset_index(drop=True)


def build_video_inventory_df() -> pd.DataFrame:
    rows: list[dict] = []
    for bucket_name, root in (("selected", SELECTED_ROOT), ("nonselected", NONSELECTED_ROOT)):
        patient_dirs = sorted(
            [path for path in root.iterdir() if path.is_dir() and path.name.startswith("p")],
            key=lambda path: natural_suffix(path.name),
        )
        for patient_dir in patient_dirs:
            lesion_videos = set(strip_lines(patient_dir / "lesionVideos.txt")) if bucket_name == "selected" else set()
            nonlesion_videos = set(strip_lines(patient_dir / "nonlesionVideos.txt")) if bucket_name == "selected" else set()
            video_dirs = sorted(
                [path for path in patient_dir.iterdir() if path.is_dir() and path.name.startswith("v")],
                key=lambda path: natural_suffix(path.name),
            )
            for video_dir in video_dirs:
                frame_paths = sorted((video_dir / "input").glob("*.png"))
                selected_frame_files = sorted(video_dir.glob("*_selectedFrames.txt"))
                selected_frame_ids = strip_lines(selected_frame_files[0]) if selected_frame_files else []
                gt_dir = video_dir / "groundtruth"
                video_key = f"{patient_dir.name}_{video_dir.name}"
                rows.append(
                    {
                        "bucket": bucket_name,
                        "model_split": expanded_video_to_split.get(video_key),
                        "patient": patient_dir.name,
                        "video": video_dir.name,
                        "video_key": video_key,
                        "frame_count": len(frame_paths),
                        "selected_frame_count": len(selected_frame_ids),
                        "lesion_flag": video_dir.name in lesion_videos,
                        "nonlesion_flag": video_dir.name in nonlesion_videos,
                        "has_groundtruth": gt_dir.exists(),
                        "projection_group": projection_lookup.get(video_key, "missing_projection")
                        if bucket_name == "selected"
                        else "not_applicable",
                    }
                )
    return pd.DataFrame(rows).sort_values(["bucket", "patient", "video"]).reset_index(drop=True)


def build_raw_selected_frame_df() -> pd.DataFrame:
    rows: list[dict] = []
    selected_patient_dirs = sorted(
        [path for path in SELECTED_ROOT.iterdir() if path.is_dir() and path.name.startswith("p")],
        key=lambda path: natural_suffix(path.name),
    )
    for patient_dir in selected_patient_dirs:
        lesion_videos = set(strip_lines(patient_dir / "lesionVideos.txt"))
        nonlesion_videos = set(strip_lines(patient_dir / "nonlesionVideos.txt"))
        video_dirs = sorted(
            [path for path in patient_dir.iterdir() if path.is_dir() and path.name.startswith("v")],
            key=lambda path: natural_suffix(path.name),
        )
        for video_dir in video_dirs:
            video_key = f"{patient_dir.name}_{video_dir.name}"
            assert video_key in expanded_video_to_split, f"Selected video missing from expanded manifest: {video_key}"
            selected_frame_files = sorted(video_dir.glob("*_selectedFrames.txt"))
            assert len(selected_frame_files) == 1, f"Expected exactly one *_selectedFrames.txt in {video_dir}"
            selected_frame_ids = strip_lines(selected_frame_files[0])
            gt_dir = video_dir / "groundtruth"
            annotation_map = {}
            if gt_dir.exists():
                for gt_path in sorted(gt_dir.glob("*.txt")):
                    if "groundTruthTable" in gt_path.name:
                        continue
                    annotation_map[gt_path.stem] = gt_path
            for frame_stem in selected_frame_ids:
                image_path = resolve_frame_image(video_dir / "input", frame_stem)
                width, height = read_png_size(image_path)
                assert (width, height) == (512, 512), f"Unexpected image size for {image_path}: {(width, height)}"
                rows.append(
                    {
                        "split": expanded_video_to_split[video_key],
                        "patient": patient_dir.name,
                        "video": video_dir.name,
                        "video_key": video_key,
                        "frame_stem": frame_stem,
                        "prepared_stem": f"cadica_{frame_stem}",
                        "image_path": str(image_path),
                        "image_hash": md5_file(image_path),
                        "has_annotation": frame_stem in annotation_map,
                        "annotation_path": str(annotation_map[frame_stem]) if frame_stem in annotation_map else None,
                        "annotation_count": count_nonempty_lines(annotation_map[frame_stem]) if frame_stem in annotation_map else 0,
                        "lesion_flag": video_dir.name in lesion_videos,
                        "nonlesion_flag": video_dir.name in nonlesion_videos,
                        "projection_group": projection_lookup.get(video_key, "missing_projection"),
                    }
                )
    frame_df = pd.DataFrame(rows).sort_values(["split", "patient", "video", "frame_stem"]).reset_index(drop=True)
    assert not frame_df.empty, "raw_selected_frame_df is empty"
    return frame_df


def build_prepared_frame_df() -> pd.DataFrame:
    columns = [
        "split",
        "prepared_stem",
        "patient",
        "video",
        "video_key",
        "frame_stem",
        "image_path",
        "label_path",
        "has_image",
        "has_label",
        "label_count",
        "image_hash",
    ]
    if not prepared_dataset_available:
        return pd.DataFrame(columns=columns)

    rows: list[dict] = []
    for split in SPLITS:
        image_dir = PREPARED_DATASET_ROOT / "images" / split
        label_dir = PREPARED_DATASET_ROOT / "labels" / split
        assert image_dir.exists(), f"Missing prepared image directory: {image_dir}"
        assert label_dir.exists(), f"Missing prepared label directory: {label_dir}"
        image_map = {path.stem: path for path in sorted(image_dir.glob("*")) if path.is_file()}
        label_map = {path.stem: path for path in sorted(label_dir.glob("*.txt")) if path.is_file()}
        for prepared_stem in sorted(set(image_map) | set(label_map)):
            match = PREPARED_STEM_RE.match(prepared_stem)
            patient = match.group(1) if match else None
            video = match.group(2) if match else None
            frame_stem = match.group(3) if match else None
            image_path = image_map.get(prepared_stem)
            label_path = label_map.get(prepared_stem)
            rows.append(
                {
                    "split": split,
                    "prepared_stem": prepared_stem,
                    "patient": patient,
                    "video": video,
                    "video_key": f"{patient}_{video}" if patient and video else None,
                    "frame_stem": frame_stem,
                    "image_path": str(image_path) if image_path else None,
                    "label_path": str(label_path) if label_path else None,
                    "has_image": image_path is not None,
                    "has_label": label_path is not None,
                    "label_count": count_nonempty_lines(label_path) if label_path else np.nan,
                    "image_hash": md5_file(image_path) if image_path else None,
                }
            )
    prepared_df = pd.DataFrame(rows).sort_values(["split", "prepared_stem"]).reset_index(drop=True)
    return prepared_df


def build_metadata_df() -> pd.DataFrame:
    metadata = pd.read_excel(METADATA_PATH)
    metadata["patient_int"] = metadata["Patient ID"].astype(int)
    metadata["patient"] = "p" + metadata["patient_int"].astype(str)
    return metadata.sort_values("patient_int").reset_index(drop=True)


def build_duplicate_groups(frame_df: pd.DataFrame, *, split_column: str) -> pd.DataFrame:
    if frame_df.empty:
        return pd.DataFrame(
            columns=[
                "image_hash",
                "occurrences",
                "unique_splits",
                "unique_patients",
                "unique_videos",
                "members",
            ]
        )
    rows: list[dict] = []
    for digest, group in frame_df.groupby("image_hash", dropna=True):
        if len(group) <= 1:
            continue
        members = group[[split_column, "patient", "video", "frame_stem"]].to_dict("records")
        rows.append(
            {
                "image_hash": digest,
                "occurrences": int(len(group)),
                "unique_splits": int(group[split_column].nunique()),
                "unique_patients": int(group["patient"].nunique()),
                "unique_videos": int(group["video"].nunique()),
                "members": members,
            }
        )
    if not rows:
        return pd.DataFrame(
            columns=[
                "image_hash",
                "occurrences",
                "unique_splits",
                "unique_patients",
                "unique_videos",
                "members",
            ]
        )
    return pd.DataFrame(rows).sort_values(
        ["occurrences", "unique_splits", "unique_patients", "unique_videos"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)


video_inventory_df = build_video_inventory_df()
raw_selected_frame_df = build_raw_selected_frame_df()
prepared_frame_df = build_prepared_frame_df()
metadata_df = build_metadata_df()
prepared_duplicate_groups_df = build_duplicate_groups(prepared_frame_df.loc[prepared_frame_df["has_image"]].copy(), split_column="split")
raw_duplicate_groups_df = build_duplicate_groups(raw_selected_frame_df.copy(), split_column="split")

table_summary_df = pd.DataFrame(
    [
        {"table": "manifest_split_df", "rows": len(manifest_split_df), "columns": len(manifest_split_df.columns)},
        {"table": "raw_selected_frame_df", "rows": len(raw_selected_frame_df), "columns": len(raw_selected_frame_df.columns)},
        {"table": "prepared_frame_df", "rows": len(prepared_frame_df), "columns": len(prepared_frame_df.columns)},
        {"table": "metadata_df", "rows": len(metadata_df), "columns": len(metadata_df.columns)},
        {"table": "video_inventory_df", "rows": len(video_inventory_df), "columns": len(video_inventory_df.columns)},
    ]
)

table_summary_df


,table,rows,columns
0,manifest_split_df,382,4
1,raw_selected_frame_df,6126,14
2,prepared_frame_df,6126,12
3,metadata_df,42,22
4,video_inventory_df,668,11


## Canonical Tables

The tables below are the audit foundation. The prepared dataset checks are conditional: if `datasets/cadica/derived/yolo26_selected_seed42` is not present, the notebook records that explicitly instead of silently skipping those checks.


In [4]:
display(Markdown("### `manifest_split_df`"))
display(manifest_split_df.head(10))

display(Markdown("### `raw_selected_frame_df`"))
display(raw_selected_frame_df.head(10))

display(Markdown("### `prepared_frame_df`"))
display(prepared_frame_df.head(10) if not prepared_frame_df.empty else pd.DataFrame([{"note": "Prepared dataset not available in this environment."}]))

display(Markdown("### `metadata_df`"))
display(metadata_df.head(10))


### `manifest_split_df`

,split,patient,video,selected_video
0,test,p18,v10,p18_v10
1,test,p18,v12,p18_v12
2,test,p18,v13,p18_v13
3,test,p18,v15,p18_v15
4,test,p18,v2,p18_v2
5,test,p18,v3,p18_v3
6,test,p18,v4,p18_v4
7,test,p18,v5,p18_v5
8,test,p18,v6,p18_v6
9,test,p18,v7,p18_v7


### `raw_selected_frame_df`

,split,patient,video,video_key,frame_stem,prepared_stem,image_path,image_hash,has_annotation,annotation_path,annotation_count,lesion_flag,nonlesion_flag,projection_group
0,test,p18,v10,p18_v10,p18_v10_00011,cadica_p18_v10_00011,/Users/iwosmura/projects/angio-demo/Angiograph...,6050417bad4f2026e8d91955f4a3eebe,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
1,test,p18,v10,p18_v10,p18_v10_00012,cadica_p18_v10_00012,/Users/iwosmura/projects/angio-demo/Angiograph...,e7fc6e0cda3323f973af1eaff4572490,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
2,test,p18,v10,p18_v10,p18_v10_00013,cadica_p18_v10_00013,/Users/iwosmura/projects/angio-demo/Angiograph...,bab2fa1f6d6fecfdec24626e1ab817bb,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
3,test,p18,v10,p18_v10,p18_v10_00014,cadica_p18_v10_00014,/Users/iwosmura/projects/angio-demo/Angiograph...,61ab29e2061cddfef1363983ffecde46,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
4,test,p18,v10,p18_v10,p18_v10_00015,cadica_p18_v10_00015,/Users/iwosmura/projects/angio-demo/Angiograph...,cf2873b0d03886fec78e31a55bdc5088,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
5,test,p18,v10,p18_v10,p18_v10_00016,cadica_p18_v10_00016,/Users/iwosmura/projects/angio-demo/Angiograph...,3708002eac6082cadda567f7ee4c4ebe,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
6,test,p18,v10,p18_v10,p18_v10_00017,cadica_p18_v10_00017,/Users/iwosmura/projects/angio-demo/Angiograph...,9baaab2a3f17590f1de45bad5bd25568,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
7,test,p18,v10,p18_v10,p18_v10_00018,cadica_p18_v10_00018,/Users/iwosmura/projects/angio-demo/Angiograph...,43a191cceb1542ded322b3e4d7b76a93,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
8,test,p18,v10,p18_v10,p18_v10_00019,cadica_p18_v10_00019,/Users/iwosmura/projects/angio-demo/Angiograph...,5f7a6998592fbbe7bf7d661f2f9e43c9,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA
9,test,p18,v10,p18_v10,p18_v10_00020,cadica_p18_v10_00020,/Users/iwosmura/projects/angio-demo/Angiograph...,5f67efebf0dc06b0a703c5690a8b562a,True,/Users/iwosmura/projects/angio-demo/Angiograph...,1,True,False,videosRCA


### `prepared_frame_df`

,split,prepared_stem,patient,video,video_key,frame_stem,image_path,label_path,has_image,has_label,label_count,image_hash
0,test,cadica_p18_v10_00011,p18,v10,p18_v10,00011,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,6050417bad4f2026e8d91955f4a3eebe
1,test,cadica_p18_v10_00012,p18,v10,p18_v10,00012,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,e7fc6e0cda3323f973af1eaff4572490
2,test,cadica_p18_v10_00013,p18,v10,p18_v10,00013,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,bab2fa1f6d6fecfdec24626e1ab817bb
3,test,cadica_p18_v10_00014,p18,v10,p18_v10,00014,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,61ab29e2061cddfef1363983ffecde46
4,test,cadica_p18_v10_00015,p18,v10,p18_v10,00015,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,cf2873b0d03886fec78e31a55bdc5088
5,test,cadica_p18_v10_00016,p18,v10,p18_v10,00016,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,3708002eac6082cadda567f7ee4c4ebe
6,test,cadica_p18_v10_00017,p18,v10,p18_v10,00017,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,9baaab2a3f17590f1de45bad5bd25568
7,test,cadica_p18_v10_00018,p18,v10,p18_v10,00018,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,43a191cceb1542ded322b3e4d7b76a93
8,test,cadica_p18_v10_00019,p18,v10,p18_v10,00019,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,5f7a6998592fbbe7bf7d661f2f9e43c9
9,test,cadica_p18_v10_00020,p18,v10,p18_v10,00020,/Users/iwosmura/projects/angio-demo/Angiograph...,/Users/iwosmura/projects/angio-demo/Angiograph...,True,True,1,5f67efebf0dc06b0a703c5690a8b562a


### `metadata_df`

,Patient ID,Age (years),Sex,Height (m),Weight (kg),BMI,Diabetes mellitus,Evolution diabetes (years),Dyslipidemia,Smoker,High blood pressure,Another comorbidity,Kidney failure,Heart failure,Atrial fibrillation,Left ventricular ejection fraction,Clinical indication for angiogrphy,Number of vessels affected,Maximum degree of the coronary artery involvement,Later medical events,patient_int,patient
0,1,55,M,NaN,55.0,NaN,0,0.0,0,1,0,0,0,0,0,>55%,Non-ST segment elevation acute coronary syndrome,0,<20%,0,1,p1
1,2,42,M,1.65,87.0,31.955923,0,0.0,1,0,0,0,0,0,0,>55%,Non-ST segment elevation acute coronary syndrome,0,<20%,0,2,p2
2,3,42,M,1.78,136.0,42.923873,0,0.0,0,1,1,0,0,0,0,>55%,Non-ST segment elevation acute coronary syndrome,0,<20%,0,3,p3
3,4,56,F,1.60,83.0,32.421875,0,0.0,0,0,0,0,0,0,0,>55%,Non-ST segment elevation acute coronary syndrome,0,<20%,0,4,p4
4,5,59,M,NaN,88.0,NaN,1,0.0,1,0,1,0,0,0,0,>55%,ST-segment Elevation Acute Coronary Syndrome,0,<20%,non-cardiac death,5,p5
5,6,72,M,NaN,66.0,NaN,1,NaN,0,1,1,0,0,0,0,>55%,Non-ST segment elevation acute coronary syndrome,0,20-50%,0,6,p6
6,7,80,F,1.59,72.0,28.479886,0,0.0,1,1,0,0,0,0,0,>55%,Non-ST segment elevation acute coronary syndrome,0,20-50%,0,7,p7
7,8,78,F,1.56,84.0,34.516765,0,0.0,1,0,1,COPD,0,0,0,>55%,stable angina,0,20-50%,non-cardiac death,8,p8
8,9,51,F,NaN,NaN,NaN,0,0.0,0,1,0,COPD,0,0,0,<35%,Non-ST segment elevation acute coronary syndrome,0,20-50%,non-cardiac death,9,p9
9,10,68,F,1.54,135.0,56.923596,1,9.0,1,1,1,0,0,0,1,>55%,stable angina,0,20-50%,0,10,p10


In [5]:
audit_rows: list[dict] = []


def add_audit(category: str, check: str, status: str, value: object, details: str) -> None:
    audit_rows.append(
        {
            "category": category,
            "check": check,
            "status": status,
            "value": value,
            "details": details,
        }
    )


def format_overlap(values: list[str], limit: int = 6) -> str:
    if not values:
        return "none"
    preview = values[:limit]
    suffix = "" if len(values) <= limit else f" ... (+{len(values) - limit} more)"
    return ", ".join(preview) + suffix


tracked_patient_set = set(tracked_patient_to_split)
expanded_patient_set = set(expanded_patient_to_split)
all_manifest_patients = set(manifest_split_df["patient"].unique())

add_audit(
    "hard_check",
    "tracked manifest patients match expanded manifest patients",
    "pass" if tracked_patient_to_split == expanded_patient_to_split else "fail",
    len(expanded_patient_to_split),
    "Tracked patient assignments and expanded local patient assignments are identical."
    if tracked_patient_to_split == expanded_patient_to_split
    else "Tracked and expanded manifests disagree on patient-to-split assignments.",
)

for left, right in combinations(SPLITS, 2):
    left_patients = set(expanded_manifest["splits"][left]["patients"])
    right_patients = set(expanded_manifest["splits"][right]["patients"])
    patient_overlap = sorted(left_patients & right_patients)
    add_audit(
        "hard_check",
        f"patient overlap between {left} and {right}",
        "pass" if not patient_overlap else "fail",
        len(patient_overlap),
        "No patient overlap detected."
        if not patient_overlap
        else f"Overlapping patients: {format_overlap(patient_overlap)}",
    )

    left_videos = set(expanded_manifest["splits"][left]["selected_videos"])
    right_videos = set(expanded_manifest["splits"][right]["selected_videos"])
    video_overlap = sorted(left_videos & right_videos)
    add_audit(
        "hard_check",
        f"selected-video overlap between {left} and {right}",
        "pass" if not video_overlap else "fail",
        len(video_overlap),
        "No selected-video overlap detected."
        if not video_overlap
        else f"Overlapping selected videos: {format_overlap(video_overlap)}",
    )

if prepared_dataset_available:
    for split in SPLITS:
        split_df = prepared_frame_df.loc[prepared_frame_df["split"] == split].copy()
        missing_labels = split_df.loc[split_df["has_image"] & ~split_df["has_label"], "prepared_stem"].tolist()
        missing_images = split_df.loc[~split_df["has_image"] & split_df["has_label"], "prepared_stem"].tolist()
        add_audit(
            "hard_check",
            f"prepared image/label parity in {split}",
            "pass" if not missing_labels and not missing_images else "fail",
            int(len(missing_labels) + len(missing_images)),
            "Every prepared image has a matching label file and vice versa."
            if not missing_labels and not missing_images
            else (
                f"Missing labels: {format_overlap(missing_labels)}; "
                f"missing images: {format_overlap(missing_images)}"
            ),
        )

    duplicate_stems = (
        prepared_frame_df.groupby("prepared_stem").size().reset_index(name="count").query("count > 1")
    )
    add_audit(
        "hard_check",
        "prepared stem uniqueness across all splits",
        "pass" if duplicate_stems.empty else "fail",
        int(len(duplicate_stems)),
        "All prepared stems are unique across train/val/test."
        if duplicate_stems.empty
        else f"Duplicate prepared stems found: {format_overlap(duplicate_stems['prepared_stem'].tolist())}",
    )

    malformed_prepared = prepared_frame_df.loc[prepared_frame_df["patient"].isna(), "prepared_stem"].tolist()
    add_audit(
        "hard_check",
        "prepared stems follow the CADICA naming scheme",
        "pass" if not malformed_prepared else "fail",
        int(len(malformed_prepared)),
        "All prepared stems match `cadica_pX_vY_frame`."
        if not malformed_prepared
        else f"Malformed prepared stems: {format_overlap(malformed_prepared)}",
    )

    prepared_wrong_split = prepared_frame_df.loc[
        prepared_frame_df["video_key"].map(expanded_video_to_split).fillna("missing") != prepared_frame_df["split"]
    ]
    add_audit(
        "hard_check",
        "prepared files stay inside the manifest split",
        "pass" if prepared_wrong_split.empty else "fail",
        int(len(prepared_wrong_split)),
        "Every prepared row maps back to the same split in the expanded manifest."
        if prepared_wrong_split.empty
        else "Prepared rows were found outside their manifest-assigned split.",
    )

    expected_prepared = raw_selected_frame_df[["split", "prepared_stem"]].copy()
    actual_prepared = prepared_frame_df.loc[prepared_frame_df["has_image"] & prepared_frame_df["has_label"], ["split", "prepared_stem"]].copy()
    expected_keys = set(expected_prepared.itertuples(index=False, name=None))
    actual_keys = set(actual_prepared.itertuples(index=False, name=None))
    missing_from_prepared = sorted(expected_keys - actual_keys)
    unexpected_in_prepared = sorted(actual_keys - expected_keys)
    add_audit(
        "hard_check",
        "prepared dataset matches raw selected-frame membership",
        "pass" if not missing_from_prepared and not unexpected_in_prepared else "fail",
        int(len(missing_from_prepared) + len(unexpected_in_prepared)),
        "Prepared train/val/test rows match the raw selected keyframes exactly."
        if not missing_from_prepared and not unexpected_in_prepared
        else "Prepared dataset is missing expected rows or includes unexpected rows.",
    )

    prepared_cross_split_duplicates = prepared_duplicate_groups_df.query("unique_splits > 1")
    add_audit(
        "hard_check",
        "exact duplicate prepared-image hashes spanning multiple splits",
        "pass" if prepared_cross_split_duplicates.empty else "fail",
        int(len(prepared_cross_split_duplicates)),
        "No byte-identical prepared images span multiple splits."
        if prepared_cross_split_duplicates.empty
        else "Byte-identical prepared images were found across multiple splits.",
    )

    prepared_local_duplicates = prepared_duplicate_groups_df.query("unique_splits == 1")
    add_audit(
        "hard_check",
        "exact duplicate prepared-image hashes confined within one split",
        "warning" if not prepared_local_duplicates.empty else "pass",
        int(len(prepared_local_duplicates)),
        "No exact duplicate prepared-image groups were found."
        if prepared_local_duplicates.empty
        else "Exact duplicate prepared-image hashes exist, but they remain confined within a single split.",
    )
else:
    for check in (
        "prepared image/label parity",
        "prepared stem uniqueness across all splits",
        "prepared files stay inside the manifest split",
        "prepared dataset matches raw selected-frame membership",
        "exact duplicate prepared-image hashes spanning multiple splits",
    ):
        add_audit(
            "hard_check",
            check,
            "warning",
            "unavailable",
            "Prepared dataset root is missing in this environment, so prepared-split checks could not run.",
        )

raw_cross_patient_duplicates = raw_duplicate_groups_df.query("unique_patients > 1 or unique_splits > 1")
add_audit(
    "hard_check",
    "exact duplicate raw selected-keyframe hashes spanning multiple patients or splits",
    "pass" if raw_cross_patient_duplicates.empty else "fail",
    int(len(raw_cross_patient_duplicates)),
    "No byte-identical raw selected keyframes span multiple patients or splits."
    if raw_cross_patient_duplicates.empty
    else "Byte-identical raw selected keyframes were found across patients or splits.",
)

raw_local_duplicates = raw_duplicate_groups_df.query("unique_patients == 1 and unique_splits == 1")
add_audit(
    "hard_check",
    "exact duplicate raw selected-keyframe hashes confined within one patient/split",
    "warning" if not raw_local_duplicates.empty else "pass",
    int(len(raw_local_duplicates)),
    "No exact duplicate raw selected-keyframe groups were found."
    if raw_local_duplicates.empty
    else "Exact duplicate raw selected keyframes exist, but they stay within a single patient and split.",
)

metadata_patient_counts = metadata_df["patient"].value_counts()
duplicate_metadata_patients = sorted(metadata_patient_counts[metadata_patient_counts > 1].index.tolist())
metadata_patients = set(metadata_df["patient"])
missing_metadata_patients = sorted(all_manifest_patients - metadata_patients)
out_of_universe_metadata = sorted(metadata_patients - all_manifest_patients)

add_audit(
    "hard_check",
    "metadata contains exactly 42 unique patients",
    "pass" if metadata_df["patient"].nunique() == EXPECTED_PATIENTS else "fail",
    int(metadata_df["patient"].nunique()),
    f"Metadata contains {metadata_df['patient'].nunique()} unique patients.",
)
add_audit(
    "hard_check",
    "metadata has one row per patient",
    "pass" if not duplicate_metadata_patients else "fail",
    int(len(duplicate_metadata_patients)),
    "Each metadata patient id appears exactly once."
    if not duplicate_metadata_patients
    else f"Duplicate metadata patients: {format_overlap(duplicate_metadata_patients)}",
)
add_audit(
    "hard_check",
    "metadata patient ids stay inside the split universe",
    "pass" if not missing_metadata_patients and not out_of_universe_metadata else "fail",
    int(len(missing_metadata_patients) + len(out_of_universe_metadata)),
    "Metadata patient ids align with the patient ids used by the split manifests."
    if not missing_metadata_patients and not out_of_universe_metadata
    else (
        f"Missing metadata patients: {format_overlap(missing_metadata_patients)}; "
        f"unexpected metadata patients: {format_overlap(out_of_universe_metadata)}"
    ),
)

selected_video_df = video_inventory_df.loc[video_inventory_df["bucket"] == "selected"].copy()
selected_counts = selected_video_df.groupby("patient")["video_key"].nunique()
both_lesion_and_nonlesion = (
    selected_video_df.groupby("patient")[["lesion_flag", "nonlesion_flag"]]
    .max()
    .assign(flag=lambda df: df["lesion_flag"] & df["nonlesion_flag"])
)
selected_and_nonselected = (
    video_inventory_df.assign(
        selected_indicator=(video_inventory_df["bucket"] == "selected").astype(int),
        nonselected_indicator=(video_inventory_df["bucket"] == "nonselected").astype(int),
    )
    .groupby("patient")[["selected_indicator", "nonselected_indicator"]]
    .max()
)
patient_projection_counts = (
    selected_video_df.loc[selected_video_df["projection_group"] != "missing_projection"]
    .groupby("patient")["projection_group"]
    .nunique()
)
missing_projection_count = int((selected_video_df["projection_group"] == "missing_projection").sum())

add_audit(
    "risk_signal",
    "patients contributing multiple selected videos",
    "warning" if int((selected_counts > 1).sum()) else "pass",
    int((selected_counts > 1).sum()),
    "Multiple selected videos per patient mean frame-level or video-level splitting would be unsafe.",
)
add_audit(
    "risk_signal",
    "patients with both lesion and non-lesion selected videos",
    "warning" if int(both_lesion_and_nonlesion["flag"].sum()) else "pass",
    int(both_lesion_and_nonlesion["flag"].sum()),
    "A naive split could leak the same patient anatomy across positive and negative examples.",
)
add_audit(
    "risk_signal",
    "patients with both selected and nonselected videos",
    "warning" if int((selected_and_nonselected["selected_indicator"] & selected_and_nonselected["nonselected_indicator"]).sum()) else "pass",
    int((selected_and_nonselected["selected_indicator"] & selected_and_nonselected["nonselected_indicator"]).sum()),
    "The dataset has parallel curation buckets for many of the same patients.",
)
add_audit(
    "risk_signal",
    "patients represented in multiple projection groups",
    "warning" if int((patient_projection_counts >= 2).sum()) else "pass",
    int((patient_projection_counts >= 2).sum()),
    "Projection diversity increases the chance that a weak split policy leaks patient identity cues.",
)
add_audit(
    "risk_signal",
    "selected videos missing projection mappings",
    "warning" if missing_projection_count else "pass",
    missing_projection_count,
    "Missing projection metadata is a modeling caveat, not leakage by itself.",
)
add_audit(
    "risk_signal",
    "filenames embed patient and video identifiers",
    "warning",
    int(len(raw_selected_frame_df)),
    "Paths and stems encode patient/video ids, so downstream exports should avoid using filenames as model features.",
)

audit_df = pd.DataFrame(audit_rows)
hard_failures_df = audit_df.query("category == 'hard_check' and status == 'fail'").reset_index(drop=True)
confirmed_no_leak_df = audit_df.query("category == 'hard_check' and status == 'pass'").reset_index(drop=True)
risk_signal_df = audit_df.query("category == 'risk_signal'").reset_index(drop=True)
warnings_df = audit_df.query("status == 'warning'").reset_index(drop=True)

audit_df


,category,check,status,value,details
0,hard_check,tracked manifest patients match expanded manif...,pass,42,Tracked patient assignments and expanded local...
1,hard_check,patient overlap between train and val,pass,0,No patient overlap detected.
2,hard_check,selected-video overlap between train and val,pass,0,No selected-video overlap detected.
3,hard_check,patient overlap between train and test,pass,0,No patient overlap detected.
4,hard_check,selected-video overlap between train and test,pass,0,No selected-video overlap detected.
5,hard_check,patient overlap between val and test,pass,0,No patient overlap detected.
6,hard_check,selected-video overlap between val and test,pass,0,No selected-video overlap detected.
7,hard_check,prepared image/label parity in train,pass,0,Every prepared image has a matching label file...
8,hard_check,prepared image/label parity in val,pass,0,Every prepared image has a matching label file...
9,hard_check,prepared image/label parity in test,pass,0,Every prepared image has a matching label file...


## Duplicate Analysis

Duplicate groups are reported separately so we can distinguish an exact repeated frame inside one patient/video from an actual cross-split leakage event.


In [6]:
display(Markdown("### Prepared duplicate groups"))
display(
    prepared_duplicate_groups_df
    if not prepared_duplicate_groups_df.empty
    else pd.DataFrame([{"note": "No exact duplicate prepared-image groups found."}])
)

display(Markdown("### Raw selected-keyframe duplicate groups"))
display(
    raw_duplicate_groups_df
    if not raw_duplicate_groups_df.empty
    else pd.DataFrame([{"note": "No exact duplicate raw selected-keyframe groups found."}])
)


### Prepared duplicate groups

,image_hash,occurrences,unique_splits,unique_patients,unique_videos,members
0,bdce3d8f1368511a8c010eac3eb75bab,2,1,1,1,"[{'split': 'train', 'patient': 'p25', 'video':..."


### Raw selected-keyframe duplicate groups

,image_hash,occurrences,unique_splits,unique_patients,unique_videos,members
0,bdce3d8f1368511a8c010eac3eb75bab,2,1,1,1,"[{'split': 'train', 'patient': 'p25', 'video':..."


## Risk Signals, Not Proof Of Leakage

The rows below are intentionally descriptive. They explain why CADICA must be split at the patient level, but they are not counted as confirmed leakage findings.


In [7]:
risk_signal_df


,category,check,status,value,details
0,risk_signal,patients contributing multiple selected videos,warning,42,Multiple selected videos per patient mean fram...
1,risk_signal,patients with both lesion and non-lesion selec...,warning,32,A naive split could leak the same patient anat...
2,risk_signal,patients with both selected and nonselected vi...,warning,36,The dataset has parallel curation buckets for ...
3,risk_signal,patients represented in multiple projection gr...,warning,40,Projection diversity increases the chance that...
4,risk_signal,selected videos missing projection mappings,warning,16,Missing projection metadata is a modeling cave...
5,risk_signal,filenames embed patient and video identifiers,warning,6126,"Paths and stems encode patient/video ids, so d..."


In [8]:
verdict = (
    "No confirmed train/val/test leakage was detected in the current patient-level seed-42 CADICA split."
    if hard_failures_df.empty
    else "Confirmed leakage or split inconsistency was detected. Review the failing checks immediately."
)

display(Markdown(f"## Headline Verdict\n\n**{verdict}**"))

display(Markdown("### Confirmed leakage findings"))
display(
    hard_failures_df
    if not hard_failures_df.empty
    else pd.DataFrame([{"check": "none", "status": "pass", "details": "No confirmed leakage findings were detected."}])
)

display(Markdown("### Confirmed no-leak checks"))
display(confirmed_no_leak_df)

display(Markdown("### Warnings and descriptive risk signals"))
display(warnings_df)

display(
    Markdown(
        "### What would leak if the split policy changed\n\n"
        "If CADICA were split at the frame or video level instead of the patient level, the model would almost certainly "
        "see the same patient anatomy in both training and evaluation. That would let patient-specific vessel geometry, "
        "projection patterns, and repeated frames inflate validation performance without representing real generalization."
    )
)

result = {
    "verdict": verdict,
    "prepared_dataset_available": prepared_dataset_available,
    "hard_failures": int(len(hard_failures_df)),
    "hard_passes": int(len(confirmed_no_leak_df)),
    "warnings": int(len(warnings_df)),
    "raw_duplicate_groups": int(len(raw_duplicate_groups_df)),
    "prepared_duplicate_groups": int(len(prepared_duplicate_groups_df)),
    "cross_split_prepared_duplicate_groups": int(
        len(prepared_duplicate_groups_df.query("unique_splits > 1")) if not prepared_duplicate_groups_df.empty else 0
    ),
}

assert hard_failures_df.empty, (
    "Hard leakage checks failed:\n"
    + hard_failures_df[["check", "details"]].to_string(index=False)
)

result


## Headline Verdict

**No confirmed train/val/test leakage was detected in the current patient-level seed-42 CADICA split.**

### Confirmed leakage findings

,check,status,details
0,none,pass,No confirmed leakage findings were detected.


### Confirmed no-leak checks

,category,check,status,value,details
0,hard_check,tracked manifest patients match expanded manif...,pass,42,Tracked patient assignments and expanded local...
1,hard_check,patient overlap between train and val,pass,0,No patient overlap detected.
2,hard_check,selected-video overlap between train and val,pass,0,No selected-video overlap detected.
3,hard_check,patient overlap between train and test,pass,0,No patient overlap detected.
4,hard_check,selected-video overlap between train and test,pass,0,No selected-video overlap detected.
5,hard_check,patient overlap between val and test,pass,0,No patient overlap detected.
6,hard_check,selected-video overlap between val and test,pass,0,No selected-video overlap detected.
7,hard_check,prepared image/label parity in train,pass,0,Every prepared image has a matching label file...
8,hard_check,prepared image/label parity in val,pass,0,Every prepared image has a matching label file...
9,hard_check,prepared image/label parity in test,pass,0,Every prepared image has a matching label file...


### Warnings and descriptive risk signals

,category,check,status,value,details
0,hard_check,exact duplicate prepared-image hashes confined...,warning,1,"Exact duplicate prepared-image hashes exist, b..."
1,hard_check,exact duplicate raw selected-keyframe hashes c...,warning,1,"Exact duplicate raw selected keyframes exist, ..."
2,risk_signal,patients contributing multiple selected videos,warning,42,Multiple selected videos per patient mean fram...
3,risk_signal,patients with both lesion and non-lesion selec...,warning,32,A naive split could leak the same patient anat...
4,risk_signal,patients with both selected and nonselected vi...,warning,36,The dataset has parallel curation buckets for ...
5,risk_signal,patients represented in multiple projection gr...,warning,40,Projection diversity increases the chance that...
6,risk_signal,selected videos missing projection mappings,warning,16,Missing projection metadata is a modeling cave...
7,risk_signal,filenames embed patient and video identifiers,warning,6126,"Paths and stems encode patient/video ids, so d..."


### What would leak if the split policy changed

If CADICA were split at the frame or video level instead of the patient level, the model would almost certainly see the same patient anatomy in both training and evaluation. That would let patient-specific vessel geometry, projection patterns, and repeated frames inflate validation performance without representing real generalization.

{'verdict': 'No confirmed train/val/test leakage was detected in the current patient-level seed-42 CADICA split.',
 'prepared_dataset_available': True,
 'hard_failures': 0,
 'hard_passes': 19,
 'warnings': 8,
 'raw_duplicate_groups': 1,
 'prepared_duplicate_groups': 1,
 'cross_split_prepared_duplicate_groups': 0}